青年城市生存力指数（YCSI）可视化入口

本 notebook 已同步为调用最新版 `ycsi_visualization.py` 的执行入口，避免和脚本维护两套重复代码。

## 1. 路径与展示工具

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

PROJECT_ROOT = Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)


## 2. 重新生成清洗数据

运行 `clean_data.py`，生成 `data/clean/job_panel.csv`、`rent_panel.csv` 和 `city_snapshot.csv`。

In [ ]:
%run clean_data.py


## 3. 生成新版静态图和导出表

运行 `ycsi_visualization.py`，完成五维友好度指标、YCSI 评分、聚类、相关性和全部图表输出。

In [ ]:
%run ycsi_visualization.py


## 4. 评分口径检查

当前脚本已同步为 `5-100` 评分口径：底层五维得分范围为 `0.05-1.00`，YCSI 综合分范围为 `5-100`。最低城市不会再显示为 0，只表示在 15 城样本中的相对低位。

In [ ]:
scores = pd.read_csv(OUTPUT_DIR / "ycsi_city_scores.csv", encoding="utf-8-sig")
score_cols = [
    "opportunity_score",
    "housing_friendliness",
    "commute_friendliness",
    "life_score",
    "growth_score",
]

display(Markdown("### \u4e94\u7ef4\u5f97\u5206\u8303\u56f4"))
display(scores[score_cols].agg(["min", "max"]).round(4))

display(Markdown("### \u91cd\u5e86\u5e95\u5206\u68c0\u67e5"))
display(scores.loc[
    scores["city"] == "\u91cd\u5e86",
    [
        "city",
        "rank",
        "ycsi",
        "commute_friendliness",
        "life_score",
        "commute_friendliness_contribution",
        "life_score_contribution",
    ],
])


## 5. 图表清单

In [ ]:
chart_plan = OUTPUT_DIR / "chart_plan.md"
print(chart_plan.read_text(encoding="utf-8"))


## 6. 展示核心图表

In [ ]:
figures = [
    ("01_ycsi_ranking.png", "YCSI \u7efc\u5408\u6392\u540d\u4e0e\u4e94\u7ef4\u8d21\u732e"),
    ("02_ycsi_city_heat_points.png", "15\u57ce\u9752\u5e74\u57ce\u5e02\u751f\u5b58\u529b\u7a7a\u95f4\u5206\u5e03"),
    ("03_salary_rent_bubble.png", "\u57ce\u5e02\u5e73\u5747\u6708\u85aa\u4e0e\u4f4f\u623f\u8d1f\u62c5\u5173\u7cfb"),
    ("04_opportunity_pressure_quadrant.png", "\u673a\u4f1a\u6307\u6570\u4e0e\u5c45\u4f4f\u53cb\u597d\u5ea6\u56db\u8c61\u9650"),
    ("05_city_radar.png", "\u805a\u7c7b\u4ee3\u8868\u57ce\u5e02\u4e94\u7ef4\u96f7\u8fbe\u56fe"),
    ("06a_life_score_dot.png", "\u751f\u6d3b\u4fbf\u5229\u7efc\u5408\u5f97\u5206\u70b9\u56fe"),
    ("06b_poi_composition_100pct.png", "\u5404\u7c7b POI \u6784\u6210 100% \u5806\u79ef\u56fe"),
    ("07a_raw_variable_correlation_heatmap.png", "\u539f\u59cb\u53d8\u91cf\u76f8\u5173\u6027\u70ed\u529b\u56fe"),
    ("07b_dimension_correlation_heatmap.png", "\u4e94\u4e2a\u4e00\u7ea7\u6307\u6807\u76f8\u5173\u6027\u70ed\u529b\u56fe"),
    ("08_city_clusters.png", "\u4e94\u7ef4 K-Means \u57ce\u5e02\u805a\u7c7b PCA \u56fe"),
    ("09_rent_trend.png", "\u5546\u54c1\u623f\u79df\u91d1\u7edd\u5bf9\u6c34\u5e73\u4e0e\u6307\u6570\u8d8b\u52bf"),
]

for filename, title in figures:
    path = OUTPUT_DIR / filename
    display(Markdown(f"### {title}"))
    if path.exists():
        display(Image(filename=str(path)))
    else:
        display(Markdown(f"\u7f3a\u5c11\u8f93\u51fa\u6587\u4ef6\uff1a`{filename}`"))


## 7. 导出数据预览

In [ ]:
scores = pd.read_csv(OUTPUT_DIR / "ycsi_city_scores.csv", encoding="utf-8-sig")
clusters = pd.read_csv(OUTPUT_DIR / "cluster_summary.csv", encoding="utf-8-sig")

display(Markdown("### YCSI \u8bc4\u5206\u8868"))
display(scores)

display(Markdown("### \u805a\u7c7b\u6c47\u603b\u8868"))
display(clusters)


## 8. 相关性 CSV 输出

In [ ]:
correlation_files = sorted(OUTPUT_DIR.glob("07*_correlation_*.csv"))
for file in correlation_files:
    print(file.name)
